In [1]:
%reload_ext autoreload
%autoreload 2

# Imports

In [ ]:
from kret_notebook import *  # NOTE import first
from kret_lgbm._core.lgbm_nb_imports import *
from kret_lightning._core.lightning_nb_imports import *
from kret_matplotlib._core.mpl_nb_imports import *
from kret_np_pd._core.np_pd_nb_imports import *
from kret_optuna._core.optuna_nb_imports import *
from kret_polars._core.polars_nb_imports import *
from kret_rosetta._core.rosetta_nb_imports import *
from kret_sklearn._core.sklearn_nb_imports import *
from kret_torch_utils._core.torch_nb_imports import *
from kret_tqdm._core.tqdm_nb_imports import *
from kret_type_hints._core.types_nb_imports import *
from kret_utils._core.utils_nb_imports import *

# from kret_wandb._core.wandb_nb_imports import *  # NOTE this is slow to import

Loaded environment variables from /Users/Akseldkw/coding/projects_kretsinger/.env
[kret_lgbm._core.lgbm_nb_imports] Imported kret_lgbm._core.lgbm_nb_imports in 1.8535 seconds
[kret_lightning._core.lightning_nb_imports] Imported kret_lightning._core.lightning_nb_imports in 4.0240 seconds
[kret_matplotlib._core.mpl_nb_imports] Imported kret_matplotlib._core.mpl_nb_imports in 0.2941 seconds
[kret_np_pd._core.np_pd_nb_imports] Imported kret_np_pd._core.np_pd_nb_imports in 0.0007 seconds
[kret_optuna._core.optuna_nb_imports] Imported kret_optuna._core.optuna_nb_imports in 0.0003 seconds
[kret_polars._core.polars_nb_imports] Imported kret_polars._core.polars_nb_imports in 0.0961 seconds
[kret_rosetta._core.rosetta_nb_imports] Imported kret_rosetta._core.rosetta_nb_imports in 0.0000 seconds
[kret_sklearn._core.sklearn_nb_imports] Imported kret_sklearn._core.sklearn_nb_imports in 0.0889 seconds


In [ ]:
from heart_nn_loader import HeartNNLoader
from heart_nn import HeartFailureNN

# Load Data

In [ ]:
UKS_CONSTANTS.KAGGLEHUB_DIR

PosixPath('/Users/Akseldkw/coding/data_kretsinger/kagglehub')

In [ ]:
file_sub_path = "datasets/heart-failure-prediction/versions/1/heart.csv"

In [ ]:
UKS_DEFAULTS.READ_CSV_PD_DEFAULT

{'header': 'infer', 'skipinitialspace': True}

In [ ]:
df_raw: pd.DataFrame = pd.read_csv(UKS_CONSTANTS.KAGGLEHUB_DIR / file_sub_path, **UKS_DEFAULTS.READ_CSV_PD_DEFAULT)

In [ ]:
dtt(df_raw)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
,int64,object,object,int64,int64,int64,object,int64,object,float64,object,int64
252,61,M,ASY,125,292,0,ST,115,Y,0.000,Up,0
497,61,M,ASY,146,241,0,Normal,148,Y,3.000,Down,1
655,40,M,ASY,152,223,0,Normal,181,N,0.000,Up,1
671,61,M,ASY,138,166,0,LVH,125,Y,3.600,Flat,1
680,57,M,ASY,150,276,0,LVH,112,Y,0.600,Flat,1


In [ ]:
counts = df_raw.value_counts()
counts

Age  Sex  ChestPainType  RestingBP  Cholesterol  FastingBS  RestingECG  MaxHR  ExerciseAngina  Oldpeak  ST_Slope  HeartDisease
28   M    ATA            130        132          0          LVH         185    N               0.0      Up        0               1
58   M    ASY            128        216          0          LVH         131    Y               2.2      Flat      1               1
                         130        0            0          ST          100    Y               1.0      Flat      1               1
                                    263          0          Normal      140    Y               2.0      Flat      1               1
                         132        458          1          Normal      69     N               1.0      Down      0               1
                                                                                                                                 ..
50   M    ASY            150        215          0          Normal      140    Y 

In [ ]:
def load_and_clean(base_dir: Path = UKS_CONSTANTS.KAGGLEHUB_DIR, filename: str = file_sub_path):
    df_load = FunctionTransformer(func=pd.read_csv, validate=False, kw_args={})
    custom_cleanup = FunctionTransformer(func=UKS_NP_PD.data_cleanup, validate=False, kw_args={"ret": True})
    pipeline_load_and_clean = PipelinePD(
        steps=[
            ("df_load", df_load),
            ("cleanup_custom", custom_cleanup),
        ]
    )
    df = pipeline_load_and_clean.fit_transform_df(base_dir / filename)
    features, target = UKS_NP_PD.pop_label_and_drop(df, label_col="HeartDisease")
    x_train, x_val, y_train, y_val = train_test_split(features, target, test_size=0.15, random_state=0)
    return x_train, x_val, y_train, y_val

In [ ]:
x_train, x_val, y_train, y_val = load_and_clean()

In [ ]:
dtt([x_train])

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
,int64,category,category,int64,int64,bool,category,int64,bool,float64,category
403,52,M,ASY,135,0,True,Normal,128,True,2.000,Flat
585,57,M,ATA,180,285,True,ST,120,False,0.800,Flat
303,62,F,ASY,120,0,True,ST,123,True,1.700,Down
74,55,M,ASY,140,268,False,Normal,128,True,1.500,Flat
668,63,F,ATA,140,195,False,Normal,179,False,0.000,Up


In [ ]:
float_cols = UKS_NP_PD.numeric_cols(x_train)
cat_cols = UKS_NP_PD.cat_cols(x_train)
power_transformer = PowerTransformer(method="yeo-johnson", standardize=True)
one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
ordinal = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

column_transform = ColumnTransformer(
    transformers=[("scaler", power_transformer, float_cols), ("onehot", one_hot, cat_cols)],
    # transformers=[("scaler", power_transformer, float_cols), ("ordinal", ordinal, cat_cols)],
    remainder="passthrough",
    verbose_feature_names_out=False,
    verbose=True,
)
steps = [("column_transform", column_transform)]
pipeline_x = PipelinePD(steps=steps)

In [ ]:
pipeline_y = PipelinePD(steps=[("scaler", StandardScaler())])  # Scale target variable

In [ ]:
loader_heart = HeartNNLoader(UKS_CONSTANTS.KAGGLEHUB_DIR, pipeline_pd_xy=(pipeline_x, pipeline_y))

Saving hparams, ignoring ('pipeline_pd_xy',)


In [ ]:
loader_heart.hparams_initial

"data_dir": /Users/Akseldkw/coding/data_kretsinger/kagglehub
"split":    None

In [ ]:
_ = loader_heart.setup("fit")

Setting up data for stage: fit
Removed 0 rows, representing 0.00% of the data
[ColumnTransformer] ........ (1 of 2) Processing scaler, total=   0.0s
[ColumnTransformer] ........ (2 of 2) Processing onehot, total=   0.0s
Setting up data for stage: validate
Removed 0 rows, representing 0.00% of the data


In [ ]:
loader_heart.set_dataloader_args(batch_size=256, shuffle=True)

In [ ]:
out = loader_heart.train_dataloader()

In [ ]:
dtt([*loader_heart.x_y_processed])

Age 
 RestingBP 
 Cholesterol 
 MaxHR 
 Oldpeak 
 Sex_F 
 Sex_M 
 ChestPainType_ASY 
 ChestPainType_ATA 
 ChestPainType_NAP 
 ChestPainType_TA 
 FastingBS_False 
 FastingBS_True 
 RestingECG_LVH 
 RestingECG_Normal 
 RestingECG_ST 
 ExerciseAngina_False 
 ExerciseAngina_True 
 ST_Slope_Down 
 ST_Slope_Flat 
 ST_Slope_Up 
 
 
 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 float64 
 
 
 
 
 3 
 -0.579 
 0.285 
 0.366 
 -1.015 
 0.760 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 
 
 185 
 0.477 
 1.473 
 0.348 
 -1.648 
 -0.805 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 
 
 315 
 2.276 
 0.661 
 -1.722 
 -0.416 
 0.594 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 1.000 
 0.000 
 0.000 
 0.000 
 1.000 
 
 
 629 
 0.369 
 -0.248 
 0.867 
 1.039 
 -0.805 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 0.000 
 0.000 
 1.000 
 
 
 788 
 1.587 
 -0.670 
 0.348 
 -0.736 
 0.760 
 1.000 
 0.000 
 0.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 0.000 
 1.000 
 0.000 
 
 
 
 
 
 
 HeartDisease 
 
 
 
 float64 
 
 
 
 
 3 
 0.860 
 
 
 185 
 0.860 
 
 
 315 
 0.860 
 
 
 629 
 -1.163 
 
 
 788 
 -1.163

In [ ]:
x_process = loader_heart.x_y_processed[0]
y_process = loader_heart.x_y_processed[1]
x_process.shape, y_process.shape

((918, 21), (918, 1))

# Implementation

In [ ]:
dtt(x_train)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
,int64,category,category,int64,int64,bool,category,int64,bool,float64,category
868,51,M,NAP,110,175,False,Normal,123,False,0.600,Up
619,74,F,ATA,120,269,False,LVH,121,True,0.200,Up
365,64,F,ASY,200,0,False,Normal,140,True,1.000,Flat
542,54,F,ASY,138,274,False,Normal,105,True,1.500,Flat
514,43,M,ASY,122,0,False,Normal,120,False,0.500,Up


In [ ]:
from kret_lightning.trainer_defaults import Trainer___init___TypedDict

static_args: Trainer___init___TypedDict = {
    "max_epochs": 20,  # Enough to see trends, not full convergence
    "limit_train_batches": 0.5,  # Use 50% of training data per epoch
    # "limit_val_batches": 0.5,  # Use 50% of val data (need reliable signal)
    # "log_every_n_steps": 50,  # Reduce logging overhead
    "enable_model_summary": False,  # Skip summary printout each trial
    "enable_checkpointing": True,  # No checkpoints during sweep (saves I/O)
    "gradient_clip_val": 1.0,  # Stability for exploring LR ranges
    "max_time": {"minutes": 30},  # Kill runaway trials
}

In [ ]:
nn = HeartFailureNN()

Saving hparams, ignoring ()


In [ ]:
TrainerDynamicDefaults.trainer_dynamic_defaults(nn, loader_heart, logtype=None)

{'logger': None,
 'default_root_dir': PosixPath('/Users/Akseldkw/coding/data_kretsinger/lightning_logs/HeartFailureNN/v_000'),
 'callbacks': []}

In [ ]:
def objective(trial: optuna.Trial) -> float:
    # Suggest hyperparameters
    hidden_size1 = trial.suggest_int("hidden_size1", 16, 128, step=16)
    hidden_size2 = trial.suggest_int("hidden_size2", 16, 128, step=16)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    l1 = trial.suggest_float("l1", 1e-5, 1e-2, log=True)
    l2 = trial.suggest_float("l2", 1e-5, 1e-2, log=True)

    model = HeartFailureNN(
        hidden_sizes=[hidden_size1, hidden_size2],
        dropout_rate=dropout_rate,
        lr=lr,
        l1_penalty=l1,
        l2_penalty=l2,
    )
    dynamic_args = TrainerDynamicDefaults.trainer_dynamic_defaults(model, loader_heart, logtype=None, trial=trial)
    trainer_args = static_args | dynamic_args

    trainer = L.Trainer(**trainer_args)  # New trainer per trial!
    assert trainer.logger is not None
    trainer.logger.log_hyperparams(model.hparams_initial)
    trainer.fit(model, datamodule=loader_heart, **TrainerStaticDefaults.TRAINER_FIT)

    return trainer.callback_metrics["val_loss"].item()

In [ ]:
study = optuna.create_study(
    study_name="heart_failure_1",
    direction="minimize",
    **OptunaDefaults.CREATE_STUDY_DEFAULTS,
)

[I 2026-01-29 01:54:01,727] Using an existing study with name 'heart_failure_1' instead of creating a new one.


In [ ]:
OptunaDefaults.OPTIM_STUDY_DEF

{'n_jobs': 1, 'gc_after_trial': True, 'show_progress_bar': True}

In [ ]:
optim_args = OptunaDefaults.study_n_trials(10) | OptunaDefaults.OPTIM_STUDY_DEF
# optim_args["n_jobs"] = 1
optim_args

{'n_trials': 10,
 'timeout': None,
 'n_jobs': 1,
 'gc_after_trial': True,
 'show_progress_bar': True}

In [ ]:
study.optimize(objective, **optim_args)

  0%|          | 0/10 [00:00<?, ?it/s]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/Akseldkw/coding/data_kretsinger/lightning_logs/HeartFailureNN/v_000 exists and is not empty.
Loading `train_dataloader` to estimate number of stepping batches.


Saving hparams, ignoring ()
Setting up data for stage: TrainerFn.FITTING
Removed 0 rows, representing 0.00% of the data
Setting up data for stage: validate
Removed 0 rows, representing 0.00% of the data


Output()

/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


[W 2026-01-29 01:54:05,001] Trial 7 failed with parameters: {'hidden_size1': 32, 'hidden_size2': 128, 'dropout_rate': 0.12365254976479578, 'lr': 5.3088666574771065e-05, 'l1': 0.00022249489650944925, 'l2': 5.8414597434243e-05} because of the following error: ValueError('Target size (torch.Size([183, 1])) must be the same as input size (torch.Size([183]))').
Traceback (most recent call last):
  File "/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/var/folders/dj/k5kk5wk90vvcnb_wphpw490r0000gn/T/ipykernel_65153/3582155911.py", line 23, in objective
    trainer.fit(model, datamodule=loader_heart, **TrainerStaticDefaults.TRAINER_FIT)
  File "/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/site-packages/lightning/pytorch/trainer/trainer.py", line 584, in fit
    call._call_and_handle_interrupt(
  File "/Users/Akseldkw/micromamba/envs/k

Traceback (most recent call last):
  File "/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
    reader_close()
  File "/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/Users/Akseldkw/micromamba/envs/kret_312/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor


ValueError: Target size (torch.Size([183, 1])) must be the same as input size (torch.Size([183]))

# Sandbox